In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import numpy as np
from sklearn.model_selection import train_test_split
import statsmodels.api as sm

In [2]:
freq = pd.read_parquet("data/raw/freMTPLfreq.parquet")
sev = pd.read_parquet("data/raw/freMTPLsev.parquet")

freq.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 413169 entries, 0 to 413168
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype   
---  ------     --------------   -----   
 0   PolicyID   413169 non-null  category
 1   ClaimNb    413169 non-null  int32   
 2   Exposure   413169 non-null  float64 
 3   Power      413169 non-null  category
 4   CarAge     413169 non-null  int32   
 5   DriverAge  413169 non-null  int32   
 6   Brand      413169 non-null  category
 7   Gas        413169 non-null  category
 8   Region     413169 non-null  category
 9   Density    413169 non-null  int32   
dtypes: category(5), float64(1), int32(4)
memory usage: 31.9 MB


In [3]:
X=freq[['Exposure', 'CarAge', 'DriverAge',
       'Brand', 'Gas', 'Region', 'Density']]
y=freq['ClaimNb']

# Encodage des variables catégorielles
X = pd.get_dummies(X, drop_first=True)
display(X)

# Split 75% / 25%
X_train, X_test, y_train, y_tes = train_test_split(X, y, test_size=0.25, random_state=42)


,Exposure,CarAge,DriverAge,Density,Brand_Japanese (except Nissan) or Korean,"Brand_Mercedes, Chrysler or BMW","Brand_Opel, General Motors or Ford",Brand_other,"Brand_Renault, Nissan or Citroen","Brand_Volkswagen, Audi, Skoda or Seat",Gas_Regular,Region_Basse-Normandie,Region_Bretagne,Region_Centre,Region_Haute-Normandie,Region_Ile-de-France,Region_Limousin,Region_Nord-Pas-de-Calais,Region_Pays-de-la-Loire,Region_Poitou-Charentes
0,0.090000,0,46,76,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,0.840000,0,46,76,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,0.520000,2,38,3003,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False
3,0.450000,2,38,3003,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False
4,0.150000,0,41,60,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413164,0.002740,0,29,2471,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
413165,0.005479,0,29,5360,True,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False
413166,0.005479,0,49,5360,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
413167,0.002740,0,41,9850,True,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False


Poisson est le choix classique pour ClaimNb (variable de comptage)

Sélection des variables AIC
 1. Commencer avec toutes les variables
 2. Retirer la variable avec la plus grande p-value > 0.05
 3. Recalculer le modèle et l'AIC
 4. Répéter jusqu'à ce que toutes les variables aient un effet significatif


In [4]:
# Ajouter l'intercept
X_train = X_train.astype(float)
X_train_const = sm.add_constant(X_train)

# GLM Poisson Modèle initial
glm_poisson = sm.GLM(y_train, X_train_const, family=sm.families.Poisson())
result_poisson = glm_poisson.fit()
display(result_poisson.summary())

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                ClaimNb   No. Observations:               309876
Model:                            GLM   Df Residuals:                   309855
Model Family:                 Poisson   Df Model:                           20
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -50426.
Date:                Wed, 26 Nov 2025   Deviance:                       77523.
Time:                        12:54:25   Pearson chi2:                 3.25e+05
No. Iterations:                     7   Pseudo R-squ. (CS):           0.007970
Covariance Type:            nonrobust                                         
============================================================================================================
                                               coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------
const                                       -3.5041      0.065    -54.066      0.000      -3.631      -3.377
Exposure                                     1.2095      0.029     42.412      0.000       1.154       1.265
CarAge                                      -0.0090      0.002     -4.958      0.000      -0.013      -0.005
DriverAge                                   -0.0075      0.001    -11.205      0.000      -0.009      -0.006
Density                                   1.461e-05   2.24e-06      6.520      0.000    1.02e-05     1.9e-05
Brand_Japanese (except Nissan) or Korean    -0.3735      0.052     -7.241      0.000      -0.475      -0.272
Brand_Mercedes, Chrysler or BMW              0.0233      0.059      0.398      0.691      -0.091       0.138
Brand_Opel, General Motors or Ford           0.0504      0.051      0.991      0.322      -0.049       0.150
Brand_other                                 -0.0451      0.071     -0.635      0.525      -0.184       0.094
Brand_Renault, Nissan or Citroen            -0.0916      0.045     -2.059      0.040      -0.179      -0.004
Brand_Volkswagen, Audi, Skoda or Seat       -0.0191      0.053     -0.363      0.717      -0.122       0.084
Gas_Regular                                 -0.1097      0.019     -5.828      0.000      -0.147      -0.073
Region_Basse-Normandie                      -0.0293      0.067     -0.439      0.661      -0.160       0.102
Region_Bretagne                              0.0525      0.045      1.159      0.246      -0.036       0.141
Region_Centre                               -0.0342      0.040     -0.862      0.389      -0.112       0.043
Region_Haute-Normandie                      -0.2345      0.088     -2.667      0.008      -0.407      -0.062
Region_Ile-de-France                         0.1116      0.047      2.380      0.017       0.020       0.203
Region_Limousin                              0.2812      0.088      3.206      0.001       0.109       0.453
Region_Nord-Pas-de-Calais                    0.0527      0.052      1.008      0.313      -0.050       0.155
Region_Pays-de-la-Loire                      0.0782      0.047      1.681      0.093      -0.013       0.169
Region_Poitou-Charentes                      0.0679      0.055      1.233      0.218      -0.040       0.176
============================================================================================================
"""

In [45]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

#Sélection des variables

# Ajouter l'intercept
X_train = X_train.astype(float)
X_train=X_train.drop(columns=['Region_Basse-Normandie', 'Brand_Mercedes, Chrysler or BMW','Brand_Volkswagen, Audi, Skoda or Seat','Region_Centre','Brand_other','Brand_Opel, General Motors or Ford'])
X_train_const = sm.add_constant(X_train)

# GLM Poisson Modèle initial
glm_poisson = sm.GLM(y_train, X_train_const, family=sm.families.Poisson())
result_poisson = glm_poisson.fit()
display(result_poisson.summary())

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                ClaimNb   No. Observations:               309876
Model:                            GLM   Df Residuals:                   309861
Model Family:                 Poisson   Df Model:                           14
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -50428.
Date:                Tue, 25 Nov 2025   Deviance:                       77528.
Time:                        23:51:32   Pearson chi2:                 3.25e+05
No. Iterations:                     7   Pseudo R-squ. (CS):           0.007955
Covariance Type:            nonrobust                                         
============================================================================================================
                                               coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------
const                                       -3.5207      0.040    -86.956      0.000      -3.600      -3.441
Exposure                                     1.2077      0.028     42.505      0.000       1.152       1.263
CarAge                                      -0.0090      0.002     -5.006      0.000      -0.013      -0.006
DriverAge                                   -0.0075      0.001    -11.205      0.000      -0.009      -0.006
Density                                   1.462e-05   2.24e-06      6.535      0.000    1.02e-05     1.9e-05
Brand_Japanese (except Nissan) or Korean    -0.3820      0.033    -11.460      0.000      -0.447      -0.317
Brand_Renault, Nissan or Citroen            -0.1041      0.020     -5.095      0.000      -0.144      -0.064
Gas_Regular                                 -0.1095      0.019     -5.837      0.000      -0.146      -0.073
Region_Bretagne                              0.0822      0.030      2.780      0.005       0.024       0.140
Region_Haute-Normandie                      -0.2065      0.081     -2.534      0.011      -0.366      -0.047
Region_Ile-de-France                         0.1380      0.035      3.982      0.000       0.070       0.206
Region_Limousin                              0.3087      0.081      3.806      0.000       0.150       0.468
Region_Nord-Pas-de-Calais                    0.0812      0.041      2.003      0.045       0.002       0.161
Region_Pays-de-la-Loire                      0.1076      0.032      3.389      0.001       0.045       0.170
Region_Poitou-Charentes                      0.0974      0.043      2.255      0.024       0.013       0.182
============================================================================================================
"""

In [46]:
from sklearn.metrics import mean_squared_error

X_test = X_test.astype(float)
X_test=X_test.drop(columns=['Region_Basse-Normandie', 'Brand_Mercedes, Chrysler or BMW','Brand_Volkswagen, Audi, Skoda or Seat','Region_Centre','Brand_other','Brand_Opel, General Motors or Ford'])
X_test_const = sm.add_constant(X_test)
y_pred_test = result_poisson.predict(X_test_const)


mse_val = mean_squared_error(y_test, y_pred_test)
print("MSE sur validation :", mse_val)


MSE sur validation : 0.042114345817809734


## Coût des sinistres

In [47]:
#Combiner les 2 bases de données

freq['PolicyID'] = freq['PolicyID'].astype(str)
sev['PolicyID'] = sev['PolicyID'].astype(str)

merged = pd.merge(freq, sev, on='PolicyID', how='left')
num_cols = merged.select_dtypes(include=['float', 'int']).columns
merged[num_cols] = merged[num_cols].fillna(0)
display(merged)


,PolicyID,ClaimNb,Exposure,Power,CarAge,DriverAge,Brand,Gas,Region,Density,ClaimAmount
0,1,0,0.090000,g,0,46,Japanese (except Nissan) or Korean,Diesel,Aquitaine,76,0.0
1,2,0,0.840000,g,0,46,Japanese (except Nissan) or Korean,Diesel,Aquitaine,76,0.0
2,3,0,0.520000,f,2,38,Japanese (except Nissan) or Korean,Regular,Nord-Pas-de-Calais,3003,0.0
3,4,0,0.450000,f,2,38,Japanese (except Nissan) or Korean,Regular,Nord-Pas-de-Calais,3003,0.0
4,5,0,0.150000,g,0,41,Japanese (except Nissan) or Korean,Diesel,Pays-de-la-Loire,60,0.0
...,...,...,...,...,...,...,...,...,...,...,...
413955,413165,0,0.002740,j,0,29,Japanese (except Nissan) or Korean,Diesel,Ile-de-France,2471,0.0
413956,413166,0,0.005479,d,0,29,Japanese (except Nissan) or Korean,Regular,Ile-de-France,5360,0.0
413957,413167,0,0.005479,k,0,49,Japanese (except Nissan) or Korean,Diesel,Ile-de-France,5360,0.0
413958,413168,0,0.002740,d,0,41,Japanese (except Nissan) or Korean,Regular,Ile-de-France,9850,0.0
